In [ ]:
import numpy as np
from numpy import linalg as npl
import matplotlib.pyplot as plt

# RMS: Root Mean Square

RMS (intensity) tells you how much activity, variation, or error is present in the system. If the arithmetic mean is the "location," the RMS is the "magnitude". It gives you a single number that summarizes the overall size of the values in a vector, regardless of their direction (positive or negative).

In machine learning, we often treat a single row of data (a "record") as a vector in space.
- Interpretation: The "intensity" is the distance of that record from the origin (0,0,0...).
- Application: In Anomaly Detection, a data point with a very high RMS across its features is often "intense" in a way that suggests it's an outlier. It is "further away" from the baseline than typical points.

### RMS vs. Standard Deviation

You might notice that RMS looks a lot like Standard Deviation (σ).
Standard Deviation measures intensity relative to the mean (how spread out the data is).
RMS measures intensity relative to zero (the total magnitude).
If your data is centered at zero, RMS and Standard Deviation are exactly the same.


# 3.1 Norm

Properties:
1. Nonnegative Homogeneity: ||Bx|| = |B|||x||
2. Triangle inequality: ||x + y|| <= ||x|| + ||y||
3. Nonnegative: ||x|| >= 0
4. Definite: ||x|| = 0 iff x = 0

## Norm of a sum
Remember `RMS = ||v|| / √n`

The question is: How does `||u + v||` relate to `||u||` and `||v||`?

### Norm of block vectors

A block vector is simply a vector formed by concatenating two or more vectors. For example, if you have two vectors u and v, you can create a block vector w by concatenating them: w = [u; v]. Block vectors are special because blocks live in orthogonal subspaces.
In real-world data science, data often comes in multiple "modes" or "blocks" (e.g., demographic data, behavioral data, and transactional data). When you combine these into a single vector, you're essentially creating a block vector. The norm of this block vector can be thought of as the combined "intensity" of all the different types of data.

By forming the block vector and taking its norm, you're saying:

"Despite the heterogeneity of these measurements, they all describe the same person. I want a single number that captures the overall 'intensity' of this person's presence in my data, while preserving the ability to decompose that intensity into its constituent aspects."

This is exactly what data scientists do when building user profiles, customer representations, or patient records.

The block norm gives you the ability to treat a collection of heterogeneous vectors as one thing when that's convenient, while keeping the aspects (AOP) accessible when you need to understand or manipulate them individually. It's abstraction made mathematical.

In [ ]:
# Nonnegative homogeneity: ||Bx|| = |B|||x||
x = [2,-1,2]
npl.norm(x), np.sqrt(np.inner(x,x)), np.sqrt(sum(np.array(x)**2))

In [ ]:
# Triangle inequality: ||x + y|| <= ||x|| + ||y||
x, y = np.random.randn(10), np.random.randn(10)
npl.norm(x + y), npl.norm(x) + npl.norm(y)

In [ ]:
# Root Mean Square like norm but within sqrt divide by amount of values in vector
# Gives a standard value for magnitude to compare to other vectors
# what a "typical" value for abs(xi) would be
rms = lambda x: npl.norm(x) / np.sqrt(len(x))
t = np.linspace(0, 1, 101)  # t = 0:.01:1
x = np.cos(8 * t) - 2 * np.sin(11 * t)
np.average(x), rms(x), x

In [ ]:
plt.plot(t,x) #some signal
plt.plot(t, np.average(x)*np.ones(len(x))) #average of the signal:yellow
plt.plot(t, (np.average(x)+rms(x))*np.ones(len(x))) #average + rms:green
plt.plot(t, (np.average(x)-rms(x))*np.ones(len(x))) #average - rms:red
plt.legend(('signal','signal avg', 'avg + rms', 'avg - rms'),
           loc='upper left')

In [ ]:
# Chebyshev's gives an upper bound for
# how many of some value would be expected in a vector
cheb_bound = lambda x, a: npl.norm(x) ** 2 // a
a = 1.5
# where x is the array producing the signal above:
cheb_bound(x, a), len(x), len([i for i in x if np.abs(i) >= a])

# This says there can be *at most*:
# 79 of the 101 values in x, with an absolute value of at least 1.5
# in fact, there are 20

If Chebyshev gives you the "bound on how far vectors stray," standardization gives you the ruler to measure that distance. The z-score is the measurement; Chebyshev is the guarantee about what you'll find.

Together they form a complete picture:

- Standardization: Express everything in comparable units (σ units)
- Z-score: The actual distance measurement
- Chebyshev: The universal law governing how extreme those measurements can be

# 3.2 Distance

You can consider distance between vectors as the norm of the difference between them. ie: dist(x,y) = ||x-y||

In [ ]:
u, v, w = np.array([1.8, 2.0, -3.7, 4.7]), np.array([.6, 2.1, 1.9, -1.4]), np.array([2.0, 1.9, -4, 4.6])
npl.norm(u - v), npl.norm(u - w), npl.norm(v - w)

In [ ]:
# You can use this notion of distance to create a "nearest neighbor" function:
nearest_neighbor = lambda x, z: z[np.argmin([npl.norm(x - y) for y in z])]
z = ([2, 1], [7, 2], [5.5, 4], [4, 8], [1, 5], [9, 6])
pointa, pointb = [5, 6], [3, 3]
nearest_neighbor(np.array(pointa), z), nearest_neighbor(np.array(pointb), z)

In [ ]:
plt.scatter(*zip(*z))
n = [str(i) for i in z]
for i, txt in enumerate(n):
    plt.annotate(txt, (z[i][0], z[i][1]))
plt.scatter(pointa[0],pointa[1])
plt.scatter(pointb[0],pointb[1])

# 3.3 Standard Deviation

In [ ]:
# De-meaning:
# useful for understanding how entries of a vector deviate from the mean
# also gives us SD in terms of norm
de_mean = lambda x: x - np.average(x)
x = np.array([1, -2.2, 3])
np.average(x), de_mean(x), np.average(de_mean(x)).round()

In [ ]:
# Standard deviation is RMS of a de-meaned vector
# gives the typical amount that vector values deviate from mean
x = np.random.rand(100)
stdev = lambda x: npl.norm(x - np.average(x)) / np.sqrt(len(x))
stdev(x), np.std(x)

In [ ]:
standardize = lambda x: (x - np.average(x)) / rms(x - np.average(x))
x = np.random.rand(100)
z = standardize(x)
np.average(x), rms(x), np.average(z).round(), rms(z)
# Taking the average to 0 and RMS to 1

# 3.4 Angle

In [ ]:
# angle
ang = lambda x, y: np.arccos(np.inner(x, y) / (npl.norm(x) * npl.norm(y)))
a, b = [1, 2, -1], [2, 0, -3]
ang(a, b), ang(a, b) * (360 / (2 * np.pi))

The correlation coefficient measures how two vectors move together. When one increases, does the other tend to increase (positive correlation), decrease (negative correlation), or show no consistent pattern (zero correlation)?

When data scientists say two features "vary together," they mean:

- Positive correlation: higher values in one tend to appear with higher values in the other
- Negative correlation: higher values in one tend to appear with lower values in the other
- Zero correlation: no consistent linear pattern

This is essential for:

- Feature selection (removing redundant features)
- Understanding relationships in data
- Detecting multicollinearity in regression
- Interpreting principal components

Note that correlation is **scale-invariant**. Multiplying a vector by a constant does not change its correlation with another vector. This matches your earlier understanding of normalization—we're comparing patterns, not magnitudes.

In [ ]:
# p value: correlation coefficient is the demeaned version of an angle
def correl_coef(a,b):
    a_tilde = a - np.average(a)
    b_tilde = b - np.average(b)
    return ((np.inner(a_tilde, b_tilde))/
            (npl.norm(a_tilde)*npl.norm(b_tilde)))

a = np.array([4.4,9.4,15.4,12.4,10.4,1.4,-4.6,-5.6,-.6,7.4])
b = np.array([6.2,11.2,14.2,14.2,8.2,2.2,-3.8,-4.8,-1.8,4.2])
correl_coef(a,b), ang(a,b)*(360/(2*np.pi))

# 3.5 Complexity

In [ ]:
x, y = np.random.rand(10 ** 6), np.random.randn(10 ** 6)
%timeit correl_coef(x,y)